# Answers: 4 · Choosing molecules

Worked answers to the **Try it yourself** exercises at the end of
[`4_choosing_molecules.ipynb`](../4_choosing_molecules.ipynb). Try each one yourself first. If you're stuck, run the cell here and
compare. Each answer runs on the same real data as the lesson, so you'll see real results,
not made-up ones.

The first code cell recreates just what the exercises need from the lesson, so this notebook
runs on its own.

Each cell runs several full campaigns, about 5 minutes for the whole notebook.

In [ ]:
import sys; sys.path.insert(0, "..")
import json, io, contextlib
import numpy as np
from tour import *
import run_experiment as rx
register_mini(rx)

def run(surrogate="single", backbones=("unimol2",), seed=42, acq="greedy", topk=100,
        init=210, batch=210, rounds=5):
    out = new_run_dir()
    which = ["--backbone", backbones[0]] if surrogate == "single" else ["--backbones", *backbones]
    sys.argv = ["run_experiment.py", "--dataset", "MINI", "--mode", "mve", "--surrogate", surrogate, *which,
                "--acq", acq, "--init-size", str(init), "--batch-size", str(batch), "--n-rounds", str(rounds),
                "--topk", str(topk), "--seed", str(seed), "--run-dir", str(out)]
    with contextlib.redirect_stdout(io.StringIO()):
        rx.main()
    return [h["topk_recall"] for h in json.loads((out / "history.json").read_text())]

SEEDS = [42, 43, 44]
print("ready")

ALL = ("molformer", "unimol2", "mhgged")

---
## Exercise 1: LT-All with and without MoLFormer

The exercise uses seed 42. We run 3 seeds so the answer doesn't rest on luck.

In [ ]:
for name, bbs in [("all 3", ALL), ("without MoLFormer", ("unimol2", "mhgged"))]:
    finals = [run("ltall", bbs, s)[-1] for s in SEEDS]
    print(f"LT-All, {name:18s} final recall {np.mean(finals):.3f}   seeds: {finals}")

**Answer.** Hardly at all. LT-All with all three backbones gets 0.960, without MoLFormer 0.957, the same within seed noise. Compare rank voting, where removing MoLFormer jumped recall from 0.71 to 0.96. LT-All learns from data how much each input is worth (notebook 4, section 5), so a weak backbone costs it almost nothing: it simply learns to lean on the other two. That's the practical case for a learned combination: you don't have to know in advance which backbones are bad.

---
## Exercise 2: rank voting with UCB (exploring backbone disagreement)

In [ ]:
for acq in ["greedy", "ucb"]:
    finals = [run("ensemble", ALL, s, acq=acq)[-1] for s in SEEDS]
    print(f"vote, all 3, {acq:6s} final recall {np.mean(finals):.3f}   seeds: {finals}")

**Answer.** Here exploring **does** help: UCB raises the rank vote from 0.713 to 0.787 (3 seeds, no overlap). That's the one place in the tour where UCB beats greedy, and the reason is interesting. In this vote, `sigma` is how much the three backbones *disagree*. Because MoLFormer is weak, it disagrees most on exactly the molecules the two good backbones agree are excellent, and an equal vote drags those down. The uncertainty bonus pushes them back up, partly undoing MoLFormer's damage. So UCB is acting as a repair for a broken model, not as smart exploration. The real fix is better: leaving MoLFormer out (0.963), or a learned combination (LT-All 0.960), beats both.

---
## Exercise 3: add MHG-GED alone to the table

In [ ]:
results = {}
for name, args in [("Uni-Mol2 alone", ("single", ("unimol2",))), ("MHG-GED alone", ("single", ("mhgged",))),
                   ("MoLFormer alone", ("single", ("molformer",)))]:
    results[name] = [run(*args, seed=s)[-1] for s in SEEDS]
import pandas as pd
pd.DataFrame({n: {"mean recall": np.mean(r), "spread": np.std(r)} for n, r in results.items()}).T.round(3)

**Answer.** MHG-GED alone gets 0.937, the same as Uni-Mol2 alone (0.940) within noise, and MoLFormer alone gets 0.290. Put next to notebook 4's table, two good single backbones (≈0.94) are only slightly behind the best combinations (≈0.96), and far ahead of the naive vote that includes MoLFormer (0.71). Combining backbones adds a little on this pool. The big wins come from not letting a weak backbone in, or from a model that learns to ignore it.